# 🔧 Fine-Tune CLIP on Flickr30k

**Goal**: Fine-tune OpenAI CLIP (ViT-B/32) on Flickr30k image-caption pairs using symmetric contrastive loss.

**Expected outcome**: +8-15% improvement in Recall@1 on Flickr30k retrieval task.

**Requirements**: Google Colab with **GPU runtime** (T4 minimum, A100 preferred)

> ⚠️ **First**: Go to **Runtime → Change runtime type → GPU (T4)** before running any cells!

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install -q transformers datasets pillow torch torchvision
!pip install -q matplotlib tqdm

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    print("\n" + "="*60)
    print("⚠️  NO GPU DETECTED!")
    print("   Go to: Runtime → Change runtime type → GPU (T4)")
    print("   Then: Runtime → Restart runtime")
    print("="*60)
    device = 'cpu'  # Will work but very slow

print(f"\nUsing device: {device}")

## 2. Mount Google Drive & Configure Paths

Upload your Flickr30k dataset to Google Drive before running.

Expected structure:
```
MyDrive/
  flickr30k/
    flickr30k_images/   ← folder with .jpg files
    captions.txt        ← pipe-delimited captions file
```

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# === CONFIGURE THESE PATHS TO MATCH YOUR DRIVE ===
DATASET_DIR = '/content/drive/MyDrive/flickr30_data/flickr30k_images'
CAPTIONS_FILE = '/content/drive/MyDrive/flickr30_data/captions.txt'
CHECKPOINT_DIR = '/content/drive/MyDrive/clip_checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Verify paths exist
for label, path in [('Images', DATASET_DIR), ('Captions', CAPTIONS_FILE)]:
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ MISSING'
    print(f"{status} {label}: {path}")
    if not exists:
        print(f"   → Please upload your dataset to: {path}")

print(f"\nCheckpoints will be saved to: {CHECKPOINT_DIR}")

## 3. Dataset Class

In [ ]:
import csv
import random
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader


class Flickr30kDataset(Dataset):
    """
    Dataset for Flickr30k image-caption pairs.
    Uses ALL 5 captions per image for richer training signal.
    """

    def __init__(self, images_dir, captions_file, processor, split='train', val_ratio=0.1):
        self.images_dir = Path(images_dir)
        self.processor = processor
        self.pairs = []

        all_pairs = []
        with open(captions_file, 'r', encoding='utf-8') as f:
            reader = csv.reader(f, delimiter='|')
            try:
                header = next(reader)  # skip header
            except StopIteration:
                pass

            for row in reader:
                if len(row) >= 3:
                    img_name = row[0].strip()
                    caption = row[2].strip()
                elif len(row) >= 2:
                    img_name = row[0].strip()
                    caption = row[1].strip()
                else:
                    continue

                img_path = self.images_dir / img_name
                if img_path.exists() and len(caption) > 5:
                    all_pairs.append((str(img_path), caption))

        # Reproducible train/val split
        random.seed(42)
        random.shuffle(all_pairs)
        split_idx = int(len(all_pairs) * (1 - val_ratio))

        if split == 'train':
            self.pairs = all_pairs[:split_idx]
        else:
            self.pairs = all_pairs[split_idx:]

        print(f"[{split}] Loaded {len(self.pairs):,} image-caption pairs")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, caption = self.pairs[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        inputs = self.processor(
            text=caption,
            images=image,
            return_tensors='pt',
            padding='max_length',
            truncation=True,
            max_length=77
        )
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
        }

## 4. Load Model & Processor

In [ ]:
from transformers import CLIPModel, CLIPProcessor

# Choose model (keep ViT-B/32 to match your local project)
MODEL_NAME = 'openai/clip-vit-base-patch32'

print(f"Loading model: {MODEL_NAME}")
model = CLIPModel.from_pretrained(MODEL_NAME)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

model = model.to(device)
print(f"Model loaded on: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Create Dataloaders

In [ ]:
train_dataset = Flickr30kDataset(DATASET_DIR, CAPTIONS_FILE, processor, split='train')
val_dataset = Flickr30kDataset(DATASET_DIR, CAPTIONS_FILE, processor, split='val')

BATCH_SIZE = 32  # Reduce to 16 if you get OOM errors

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=(device == 'cuda'),
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device == 'cuda'),
    drop_last=False
)

print(f"Train batches: {len(train_loader):,}")
print(f"Val batches: {len(val_loader):,}")

## 6. Training Setup

In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm


def contrastive_loss(logits_per_image, logits_per_text):
    """Symmetric cross-entropy contrastive loss (CLIP's training objective)."""
    batch_size = logits_per_image.shape[0]
    labels = torch.arange(batch_size, device=logits_per_image.device)
    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)
    return (loss_i + loss_t) / 2


# Hyperparameters
EPOCHS = 4
LEARNING_RATE = 5e-7      # Very low to prevent catastrophic forgetting
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.98),
    eps=1e-6
)

total_steps = len(train_loader) * EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=LEARNING_RATE * 0.1
)

# Mixed precision (PyTorch 2.x API — not the deprecated torch.cuda.amp)
use_amp = (device == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

print(f"Total steps: {total_steps:,}")
print(f"Mixed precision (AMP): {use_amp}")
print(f"Device: {device}")

## 7. Training Loop

In [ ]:
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience = 2
patience_counter = 0

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')

    for batch in pbar:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda', enabled=use_amp):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_loss=False
            )
            loss = contrastive_loss(outputs.logits_per_image, outputs.logits_per_text)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')

    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]'):
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            with torch.amp.autocast('cuda', enabled=use_amp):
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    return_loss=False
                )
                loss = contrastive_loss(outputs.logits_per_image, outputs.logits_per_text)
            val_loss += loss.item()

    avg_val_loss = val_loss / max(len(val_loader), 1)
    val_losses.append(avg_val_loss)

    print(f'\nEpoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}')

    # Save checkpoint to Drive
    ckpt_path = f'{CHECKPOINT_DIR}/clip_epoch_{epoch+1}'
    model.save_pretrained(ckpt_path)
    processor.save_pretrained(ckpt_path)
    print(f'Checkpoint saved: {ckpt_path}')

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_path = f'{CHECKPOINT_DIR}/clip_best'
        model.save_pretrained(best_path)
        processor.save_pretrained(best_path)
        print(f'✅ New best model! Val loss = {best_val_loss:.4f}')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'⚠️ Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)')
            break

print(f'\n🎉 Training complete! Best val loss: {best_val_loss:.4f}')

## 8. Training Loss Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(range(1, len(train_losses)+1), train_losses, 'b-o', label='Train Loss', linewidth=2)
ax.plot(range(1, len(val_losses)+1), val_losses, 'r-s', label='Val Loss', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Contrastive Loss', fontsize=12)
ax.set_title('CLIP Fine-Tuning on Flickr30k', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/training_curve.png', dpi=150)
plt.show()
print(f'Training curve saved to {CHECKPOINT_DIR}/training_curve.png')

## 9. Next Steps

After training completes:
1. The best checkpoint is saved at `clip_best/` in your Google Drive
2. Download it to your local project: `fine_tuning/checkpoints/clip_best/`
3. Update `core/config.py`:
```python
CLIP_MODEL_PATH = APP_DIR / "fine_tuning" / "checkpoints" / "clip_best"
```
4. Delete `storage/faiss.index` and re-index your images
5. Run evaluation to measure improvement

In [ ]:
# Summary
print('=' * 50)
print('  FINE-TUNING SUMMARY')
print('=' * 50)
print(f'Model:         {MODEL_NAME}')
print(f'Device:        {device}')
print(f'Epochs run:    {len(train_losses)}')
print(f'Best val loss: {best_val_loss:.4f}')
print(f'Checkpoints:   {CHECKPOINT_DIR}')
print(f'Best model:    {CHECKPOINT_DIR}/clip_best')
print('=' * 50)